### Data collecting

In [2]:
import pandas as pd
import os


df = pd.read_csv('.\\annotations_metadata.csv')

In [ ]:
text_files_dir = '.\\all_files'
texts = []

# Iterate over each row in the dataframe to read the corresponding text file
for file_id in df['file_id']:
    file_path = os.path.join(text_files_dir, f"{file_id}.txt")
    text_content = None 
    try:
        with open(file_path, 'r', encoding='utf-8-sig') as file:
            text_content = file.read()
    except Exception as e:
        print(f"Error reading file {file_id}: {e}")
    texts.append(text_content)

df_with_text = df.copy() 
df_with_text['text'] = texts
df_with_text

In [6]:
os.chdir('.\\240613_작업')

df_with_text.to_csv('supremacist_all.csv', index=False, encoding='utf-8-sig')

### Data preprocess

In [ ]:
df = pd.read_csv('supremacist_all.csv')
df

In [ ]:
df= df.drop(columns=['file_id','user_id','subforum_id', 'num_contexts'])
df

In [ ]:
df['label'].value_counts()

In [ ]:
df1= df[df['label'] == 'noHate']
df1

In [ ]:
df2 = df[df['label'] == 'hate']
df2

In [ ]:
data = pd.concat([df1,df2])
data

In [ ]:
import re

def preprocess_text(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'(?<!\.)\.{3,}(?!\.)', '', text) 
    text = re.sub(r'[^\w\s]', '', text) 
    text = re.sub(r'\b\d+(\.\d+)?', '', text) 
    text = re.sub(r'\b\d+(\.\d+)?', '', text) 
    text = re.sub(r'\d+', '', text) 
    text = re.sub(r'_', '', text) 
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'http', '', text)
    text = re.sub(r'https', '', text)
    text = re.sub(r'nt', '', text)
    text = re.sub(r'mkr', '', text)
    text = re.sub(r've', '', text)
    text = re.sub(r'don', '', text)
    text = re.sub(r'kat', '', text)
    text = re.sub(r'kabir', '', text)
    text = re.sub(r'singh', '', text)
    text = re.sub(r'bhai', '', text)
    #text = re.sub(r'th', '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text

data['prepro'] = data['text'].apply(preprocess_text)
data

In [ ]:
data['encoding'] = data['label'].apply(lambda x: 1 if x == 'hate' else 0)
data

In [ ]:
data = data.sample(frac=1, random_state=42)
data = data.drop(columns='index')
data

In [49]:
data.to_csv('supremacist_prepro.csv',index=False, encoding='utf-8-sig')

## After preprocessing

### extract BERT[CLS] vectors

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import torch

file_path = 'supremacist_prepro.csv'
save_path = 'supremacist_cls_vector.npy'
df = pd.read_csv(file_path)
df['prepro'] = df['prepro'].apply(lambda x: str(x) if pd.notnull(x) else '')
df = df[df['prepro'].apply(lambda x: len(x.split()) > 3)]
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device :{device}')
model.to(device)

def extract_cls_tokens(texts, tokenizer, model):
    cls_tokens = []
    for text in tqdm(texts, desc="Extracting CLS tokens"):
        inputs = tokenizer.encode_plus(text, add_special_tokens=True, max_length=64, padding='max_length', truncation=True, return_tensors="pt")
        with torch.inference_mode(): 
            outputs = model(**inputs)
        cls_token = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy()
        cls_tokens.append(cls_token)
    cls_tokens_array = np.vstack(cls_tokens) 
    np.save(save_path, cls_tokens_array)
    return cls_tokens

extract_cls_tokens(df['prepro'], tokenizer, model)

#### BERT[CLS] + VADER vectors

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

file_path = 'supremacist_input.csv'
save_path = 'cls+vader.npy'
df = pd.read_csv(file_path)
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
model.to(device)
analyzer = SentimentIntensityAnalyzer()

def extract_combined_embeddings(texts, tokenizer, model, analyzer):
    combined_embeddings = []
    for text in tqdm(texts, desc="Extracting combined embeddings"):
        inputs = tokenizer.encode_plus(text, add_special_tokens=True, max_length=64, padding='max_length', truncation=True, return_tensors="pt").to(device)
        with torch.inference_mode():
            outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :].cpu()
        sentiment_score = analyzer.polarity_scores(text)['compound']
        sentiment_tensor = torch.tensor([[sentiment_score]], dtype=torch.float32)
        combined_embedding = torch.cat((cls_embedding, sentiment_tensor), dim=1).numpy()
        combined_embeddings.append(combined_embedding)

    combined_embeddings_array = np.vstack(combined_embeddings)
    np.save(save_path, combined_embeddings_array)
    return combined_embeddings

extract_combined_embeddings(df['prepro'], tokenizer, model, analyzer)

#### BERT[CLS] + LIWC vectors

In [ ]:
df= pd.read_csv('.\\supremacist_liwc.csv')
df2 = np.load('supremacist_cls_vector.npy')

df = df.drop(columns=['A','B','Segment'])
dff = df.index[0]
df = df.drop(dff)
df = df.to_numpy()

final = np.concatenate((df, df2), axis=1)
np.save('cls+liwc.npy', final)

#### split train and test dataset

In [75]:
from sklearn.model_selection import train_test_split
###train/test split

df = pd.read_csv("supremacist_input.csv")
features = np.load("supremacist_cls+vader.npy")

df_train, df_test, features_train, features_test = train_test_split(
    df, features, test_size=0.2, stratify=df['encoding'], random_state=42)
df_train.to_csv("supermacist_train.csv", index=False, encoding = 'utf-8-sig')
df_test.to_csv("supermacist_test.csv", index=False, encoding = 'utf-8-sig')
np.save("vader_cls_train.npy", features_train)
np.save("vader_cls_test.npy", features_test)

print("Test dataset and features have been created and saved.")

Test dataset and features have been created and saved.


# Modeling

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
import pandas as pd
import random
import os
import torch
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

### only with BERT[CLS]

In [ ]:
class Classifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(Classifier, self).__init__()
        self.linear = nn.Linear(input_dim, num_classes)
    
    def forward(self, x):
        return self.linear(x)

def load_data(file_path, feature_path):
    df = pd.read_csv(file_path)
    labels = df['encoding'].tolist()  
    features = np.load(feature_path)
    return labels, features

def build_dataloader(X, y, batch_size):
    tensor_x = torch.tensor(X).float()
    tensor_y = torch.tensor(y).long()
    dataset = TensorDataset(tensor_x, tensor_y)
    dataloader = DataLoader(dataset, shuffle=True, batch_size=batch_size)
    return dataloader

def set_seed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, device, epochs, early_stopping_patience=3):
    best_val_loss = float('inf')
    early_stopping_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, total_train = 0, 0, 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, pred = torch.max(output, dim=1)
            train_correct += (pred == target).sum().item()
            total_train += target.size(0)
        
        train_accuracy = train_correct / total_train
        
        val_loss, val_correct, total_val = 0, 0, 0
        model.eval()
        with torch.inference_mode():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                val_loss += loss.item()
                _, pred = torch.max(output, dim=1)
                val_correct += (pred == target).sum().item()
                total_val += target.size(0)
   
        val_accuracy = val_correct / total_val
        
        print(f'Epoch {epoch+1}/{epochs}')
        print(f'Train_loss: {train_loss:.4f} - Train_acc: {train_accuracy:.4f} \nVal_loss: {val_loss:.4f} - Val_acc: {val_accuracy:.4f}')
        print('='*30)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stopping_counter = 0
            # Save the best model
            torch.save(model.state_dict(), "best_model.pth")
        else:
            early_stopping_counter += 1
            if early_stopping_counter >= early_stopping_patience:
                print("Early stopping triggered.")
                break

def process_data_and_train_model(file_path, feature_path, batch_size, epochs, lr, device, seed=42):
    set_seed(seed)  

    labels, features = load_data(file_path, feature_path)
    X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2, random_state=seed, stratify=labels)
    train_loader = build_dataloader(X_train, y_train, batch_size)
    val_loader = build_dataloader(X_val, y_val, batch_size)
    num_classes = 2  
    input_size = 768
    model = Classifier(input_dim=input_size, num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay = 1e-2)

    train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, device, epochs)

# Parameters for model training
file_path = "supremacist_train.csv"  
feature_path = "supremacist_cls_train.npy" 
batch_size = 16
epochs = 20
lr = 2e-5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

process_data_and_train_model(file_path, feature_path, batch_size, epochs, lr, device)


#### test

In [ ]:
class Classifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(Classifier, self).__init__()
        self.linear = nn.Linear(input_dim, num_classes)
    
    def forward(self, x):
        return self.linear(x)

def load_data(file_path, feature_path):
    df = pd.read_csv(file_path)
    labels = df['encoding'].tolist() 
    features = np.load(feature_path)
    return labels, features

def build_dataloader(X, y, batch_size):
    tensor_x = torch.tensor(X).float()
    tensor_y = torch.tensor(y).long()
    dataset = TensorDataset(tensor_x, tensor_y)
    dataloader = DataLoader(dataset, shuffle=True, batch_size=batch_size)
    return dataloader

def load_best_model(model_path, num_classes, input_size, device):
    model = Classifier(num_classes=num_classes, input_dim=input_size).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device)) 
    model.to(device)
    return model

def evaluate(model, data_loader, device):
    model.eval()
    total_correct, total = 0, 0
    all_predictions = []
    all_targets = []

    with torch.inference_mode():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, pred = torch.max(output, dim=1)
            total_correct += (pred == target).sum().item()
            total += target.size(0)
            all_predictions.extend(pred.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    accuracy = total_correct / total
    precision = precision_score(all_targets, all_predictions, average='macro')  
    recall = recall_score(all_targets, all_predictions, average='macro') 
    f1 = f1_score(all_targets, all_predictions, average='macro')

    return accuracy, precision, recall, f1

def test_model(test_file_path, test_feature_path, batch_size, model_path, device):
    
    test_labels, test_features = load_data(test_file_path, test_feature_path)
    test_loader = build_dataloader(test_features, test_labels, batch_size)
    num_classes = 2  
    input_size = 768 
    best_model = load_best_model(model_path, num_classes, input_size, device);
    accuracy, precision, recall, f1 = evaluate(best_model, test_loader, device)
    print(f"Test Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")

test_file_path = "supremacist_test.csv"
test_feature_path = "supremacist_cls_test.npy"
batch_size = 16
model_path = "best_model.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_model(test_file_path, test_feature_path, batch_size, model_path, device)

### BERT + CNN train/validation

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, num_classes, input_size, kernel_size=3, dropout_rate=0.2):
        super(CNNClassifier, self).__init__()
        self.conv = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=kernel_size)
        self.bn = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.pool = nn.MaxPool1d(kernel_size=kernel_size)
        self.fc_input_size = self._calculate_fc_input_size(input_size, kernel_size)
        self.hidden1 = nn.Linear(self.fc_input_size, 256)
        self.hidden1_bn = nn.BatchNorm1d(256)
        self.hidden1_relu = nn.ReLU()
        self.hidden1_dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(256, num_classes)

    def _calculate_fc_input_size(self, input_size, kernel_size):
        size = input_size
        size = (size - (kernel_size - 1) - 1) + 1
        size = size // kernel_size
        size = size * 64
        return size

    def forward(self, x):
        x = x.unsqueeze(1)        
        x = self.conv(x)           
        x = self.bn(x)             
        x = self.relu(x)           
        x = self.dropout(x)       
        x = self.pool(x)           
        x = torch.flatten(x, 1)   
        x = self.hidden1(x)        
        x = self.hidden1_bn(x)    
        x = self.hidden1_relu(x)  
        x = self.hidden1_dropout(x)
        x = self.fc(x)             
        return x
    
def load_data(file_path, feature_path):
    df = pd.read_csv(file_path)
    labels = df['encoding'].tolist()  
    features = np.load(feature_path)
    return labels, features

def build_dataloader(X, y, batch_size):
    tensor_x = torch.tensor(X).float()
    tensor_y = torch.tensor(y).long()
    dataset = TensorDataset(tensor_x, tensor_y)
    dataloader = DataLoader(dataset, shuffle=True, batch_size=batch_size)
    return dataloader

def set_seed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, device, epochs, early_stopping_patience=3):
    best_val_loss = float('inf')
    early_stopping_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, total_train = 0, 0, 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, pred = torch.max(output, dim=1)
            train_correct += (pred == target).sum().item()
            total_train += target.size(0)
        
        train_accuracy = train_correct / total_train
        
        val_loss, val_correct, total_val = 0, 0, 0
        model.eval()
        with torch.inference_mode():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                val_loss += loss.item()
                _, pred = torch.max(output, dim=1)
                val_correct += (pred == target).sum().item()
                total_val += target.size(0)
   
        val_accuracy = val_correct / total_val
        
        print(f'Epoch {epoch+1}/{epochs}')
        print(f'Train_loss: {train_loss:.4f} - Train_acc: {train_accuracy:.4f} \nVal_loss: {val_loss:.4f} - Val_acc: {val_accuracy:.4f}')
        print('='*30)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stopping_counter = 0
            torch.save(model.state_dict(), "best_model(vader+cnn).pth")
        else:
            early_stopping_counter += 1
            if early_stopping_counter >= early_stopping_patience:
                print("Early stopping triggered.")
                break

def process_data_and_train_model(file_path, feature_path, batch_size, epochs, lr, device, seed=42):
    set_seed(seed) 

    labels, features = load_data(file_path, feature_path)
    X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2, random_state=seed, stratify=labels)
    train_loader = build_dataloader(X_train, y_train, batch_size)
    val_loader = build_dataloader(X_val, y_val, batch_size)
    num_classes = 2  
    input_size = 769 #+ 81 
    model = CNNClassifier(num_classes=num_classes, input_size=input_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay = 1e-2)

    train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, device, epochs)

# Parameters for model training
file_path = "supremacist_train.csv"  
feature_path = "vader_cls_train.npy" 
batch_size = 16
epochs = 20
lr = 2e-5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

process_data_and_train_model(file_path, feature_path, batch_size, epochs, lr, device)

#### BERT + CNN test

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, num_classes, input_size, kernel_size=3, dropout_rate=0.2):
        super(CNNClassifier, self).__init__()
        self.conv = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=kernel_size)
        self.bn = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.pool = nn.MaxPool1d(kernel_size=kernel_size)
        self.fc_input_size = self._calculate_fc_input_size(input_size, kernel_size)
        self.hidden1 = nn.Linear(self.fc_input_size, 256)
        self.hidden1_bn = nn.BatchNorm1d(256)
        self.hidden1_relu = nn.ReLU()
        self.hidden1_dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(256, num_classes)

    def _calculate_fc_input_size(self, input_size, kernel_size):
        size = input_size
        size = (size - (kernel_size - 1) - 1) + 1
        size = size // kernel_size
        size = size * 64
        return size

    def forward(self, x):
        x = x.unsqueeze(1)         
        x = self.conv(x)           
        x = self.bn(x)             
        x = self.relu(x)           
        x = self.dropout(x)        
        x = self.pool(x)           
        x = torch.flatten(x, 1)   
        x = self.hidden1(x)        
        x = self.hidden1_bn(x)     
        x = self.hidden1_relu(x)   
        x = self.hidden1_dropout(x)
        x = self.fc(x)            
        return x

def load_data(file_path, feature_path):
    df = pd.read_csv(file_path)
    labels = df['encoding'].tolist() 
    features = np.load(feature_path)
    return labels, features

def build_dataloader(X, y, batch_size):
    tensor_x = torch.tensor(X).float()
    tensor_y = torch.tensor(y).long()
    dataset = TensorDataset(tensor_x, tensor_y)
    dataloader = DataLoader(dataset, shuffle=True, batch_size=batch_size)
    return dataloader

def load_best_model(model_path, num_classes, input_size, device):
    model = CNNClassifier(num_classes=num_classes, input_size=input_size).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))  
    model.to(device)
    return model

def evaluate(model, data_loader, device):
    model.eval()
    total_correct, total = 0, 0
    all_predictions = []
    all_targets = []

    with torch.inference_mode():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, pred = torch.max(output, dim=1)
            total_correct += (pred == target).sum().item()
            total += target.size(0)
            all_predictions.extend(pred.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    accuracy = total_correct / total
    precision = precision_score(all_targets, all_predictions, average='macro')  
    recall = recall_score(all_targets, all_predictions, average='macro') 
    f1 = f1_score(all_targets, all_predictions, average='macro')

    return accuracy, precision, recall, f1

def test_model(test_file_path, test_feature_path, batch_size, model_path, device):
    
    test_labels, test_features = load_data(test_file_path, test_feature_path)
    test_loader = build_dataloader(test_features, test_labels, batch_size)
    num_classes = 2  
    input_size = 768 + 81 
    best_model = load_best_model(model_path, num_classes, input_size, device)
    accuracy, precision, recall, f1 = evaluate(best_model, test_loader, device)
    print(f"Test Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")

test_file_path = "supremacist_test.csv"
test_feature_path = "liwc_cls_test.npy"
batch_size = 16
model_path = "best_model(liwc+cnn).pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_model(test_file_path, test_feature_path, batch_size, model_path, device)

### BERT + DNN train/validation

In [ ]:
class SimpleDNN(nn.Module):
    def __init__(self, input_size, num_classes, dropout_rate=0.2):
        super(SimpleDNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(256, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)
        self.fc3 = nn.Linear(64, 16)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout_rate)
        self.fc4 = nn.Linear(16, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.dropout3(x)
        x = self.fc4(x)
        return x

def load_data(file_path, feature_path):
    df = pd.read_csv(file_path)
    labels = df['encoding'].tolist() 
    features = np.load(feature_path)
    return labels, features

def build_dataloader(X, y, batch_size):
    tensor_x = torch.tensor(X).float()
    tensor_y = torch.tensor(y).long()
    dataset = TensorDataset(tensor_x, tensor_y)
    dataloader = DataLoader(dataset, shuffle=True, batch_size=batch_size)
    return dataloader

def set_seed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, device, epochs, early_stopping_patience=3):
    best_val_loss = float('inf')
    early_stopping_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, total_train = 0, 0, 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, pred = torch.max(output, dim=1)
            train_correct += (pred == target).sum().item()
            total_train += target.size(0)
        
        train_accuracy = train_correct / total_train
        
        val_loss, val_correct, total_val = 0, 0, 0
        model.eval()
        with torch.inference_mode():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                val_loss += loss.item()
                _, pred = torch.max(output, dim=1)
                val_correct += (pred == target).sum().item()
                total_val += target.size(0)
   
        val_accuracy = val_correct / total_val
        
        print(f'Epoch {epoch+1}/{epochs}')
        print(f'Train_loss: {train_loss:.4f} - Train_acc: {train_accuracy:.4f} \nVal_loss: {val_loss:.4f} - Val_acc: {val_accuracy:.4f}')
        print('='*30)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stopping_counter = 0
            torch.save(model.state_dict(), "best_model(cls+liwc+vader).pth")
        else:
            early_stopping_counter += 1
            if early_stopping_counter >= early_stopping_patience:
                print("Early stopping triggered.")
                break

def process_data_and_train_model(file_path, feature_path, batch_size, epochs, lr, device, seed=42):
    set_seed(seed)

    labels, features = load_data(file_path, feature_path)
    X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2, random_state=seed, stratify=labels)
    train_loader = build_dataloader(X_train, y_train, batch_size)
    val_loader = build_dataloader(X_val, y_val, batch_size)
    input_size = 768+81
    num_classes = 2
    model = SimpleDNN(input_size=input_size, num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, device, epochs)

# Parameters for model training
file_path = "supremacist_train.csv"  
feature_path = "liwc_cls_train.npy"  
batch_size = 16
epochs = 20
lr = 2e-5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

process_data_and_train_model(file_path, feature_path, batch_size, epochs, lr, device)

#### BERT + DNN test

In [ ]:
### test

def load_data(file_path, feature_path):
    df = pd.read_csv(file_path)
    labels = df['encoding'].tolist() 
    features = np.load(feature_path)
    return labels, features

def build_dataloader(X, y, batch_size):
    tensor_x = torch.tensor(X).float()
    tensor_y = torch.tensor(y).long()
    dataset = TensorDataset(tensor_x, tensor_y)
    dataloader = DataLoader(dataset, shuffle=True, batch_size=batch_size)
    return dataloader

def load_best_model(model_path, num_classes, input_size, device):
    model = SimpleDNN(num_classes=num_classes, input_size=input_size).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device)) 
    model.to(device)
    return model

def evaluate(model, data_loader, device):
    model.eval()
    total_correct, total = 0, 0
    all_predictions = []
    all_targets = []

    with torch.inference_mode():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, pred = torch.max(output, dim=1)
            total_correct += (pred == target).sum().item()
            total += target.size(0)
            all_predictions.extend(pred.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    accuracy = total_correct / total
    precision = precision_score(all_targets, all_predictions, average='macro')  
    recall = recall_score(all_targets, all_predictions, average='macro') 
    f1 = f1_score(all_targets, all_predictions, average='macro')

    return accuracy, precision, recall, f1

def test_model(test_file_path, test_feature_path, batch_size, model_path, device):
    test_labels, test_features = load_data(test_file_path, test_feature_path)
    test_loader = build_dataloader(test_features, test_labels, batch_size)
    num_classes = 2
    input_size = 768 + 81

    best_model = load_best_model(model_path, num_classes, input_size, device)

    accuracy, precision, recall, f1 = evaluate(best_model, test_loader, device)
    print(f"Test Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")

test_file_path = "supremacist_test.csv"
test_feature_path = "liwc_cls_test.npy"
batch_size = 16
model_path = "best_model(cls+liwc+vader).pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_model(test_file_path, test_feature_path, batch_size, model_path, device)

### BERT+LSTM

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, num_classes, input_size, hidden_size, num_layers, dropout_rate=0.2):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_rate)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        x = hn[-1, :, :]  # Take the last hidden state of the last LSTM layer
        x = self.dropout(x)
        x = self.fc(x)
        return x

def load_data(file_path, feature_path):
    df = pd.read_csv(file_path)
    labels = df['encoding'].tolist() 
    features = np.load(feature_path)
    return labels, features

def build_dataloader(X, y, batch_size):
    tensor_x = torch.tensor(X).float()
    tensor_y = torch.tensor(y).long()
    dataset = TensorDataset(tensor_x, tensor_y)
    dataloader = DataLoader(dataset, shuffle=True, batch_size=batch_size)
    return dataloader

def set_seed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, device, epochs, early_stopping_patience=3):
    best_val_loss = float('inf')
    early_stopping_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, total_train = 0, 0, 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, pred = torch.max(output, dim=1)
            train_correct += (pred == target).sum().item()
            total_train += target.size(0)
        
        train_accuracy = train_correct / total_train
        
        val_loss, val_correct, total_val = 0, 0, 0
        model.eval()
        with torch.inference_mode():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                val_loss += loss.item()
                _, pred = torch.max(output, dim=1)
                val_correct += (pred == target).sum().item()
                total_val += target.size(0)
                
        val_accuracy = val_correct / total_val
        
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_accuracy:.4f}, Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_accuracy:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stopping_counter = 0
           
            torch.save(model.state_dict(), "best_model(lstm+vader).pth")
        else:
            early_stopping_counter += 1
            if early_stopping_counter >= early_stopping_patience:
                print("Early stopping triggered.")
                break

def process_data_and_train_model(file_path, feature_path, batch_size, epochs, lr, device, seed=42):
    set_seed(seed)  

    labels, features = load_data(file_path, feature_path)
    X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2, random_state=seed, stratify=labels)
    X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
    X_val = X_val.reshape(X_val.shape[0], 1, X_val.shape[1])
    train_loader = build_dataloader(X_train, y_train, batch_size)
    val_loader = build_dataloader(X_val, y_val, batch_size)

    num_classes = 2  
    input_size = 769 #+ 81 
    hidden_size = 128
    num_layers = 2
    model = LSTMClassifier(num_classes=num_classes, input_size=input_size, hidden_size=hidden_size, num_layers=num_layers).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, device, epochs)

# Parameters for model training
file_path = "supremacist_train.csv" 
feature_path = "vader_cls_train.npy" 
batch_size = 16
epochs = 20
lr = 2e-5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

process_data_and_train_model(file_path, feature_path, batch_size, epochs, lr, device)


#### test

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, num_classes, input_size, hidden_size, num_layers, dropout_rate=0.2):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_rate)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        x = hn[-1]  # Take the last hidden state of the last LSTM layer
        x = self.dropout(x)
        x = self.fc(x)
        return x

def load_data(file_path, feature_path):
    df = pd.read_csv(file_path)
    labels = df['encoding'].tolist()
    features = np.load(feature_path)
    return labels, features

def build_dataloader(X, y, batch_size):
    tensor_x = torch.tensor(X).float().unsqueeze(1)
    tensor_y = torch.tensor(y).long()
    dataset = TensorDataset(tensor_x, tensor_y)
    dataloader = DataLoader(dataset, shuffle=True, batch_size=batch_size)
    return dataloader

def load_best_model(model_path, num_classes, input_size, device):
    model = LSTMClassifier(num_classes=num_classes, input_size=input_size, hidden_size=128, num_layers=2).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    return model

def evaluate(model, data_loader, device):
    model.eval()
    total_correct, total = 0, 0
    all_predictions = []
    all_targets = []

    with torch.inference_mode():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, pred = torch.max(output, dim=1)
            total_correct += (pred == target).sum().item()
            total += target.size(0)
            all_predictions.extend(pred.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    accuracy = total_correct / total
    precision = precision_score(all_targets, all_predictions, average='macro')
    recall = recall_score(all_targets, all_predictions, average='macro')
    f1 = f1_score(all_targets, all_predictions, average='macro')

    return accuracy, precision, recall, f1

def test_model(test_file_path, test_feature_path, batch_size, model_path, device):
    test_labels, test_features = load_data(test_file_path, test_feature_path)
    test_loader = build_dataloader(test_features, test_labels, batch_size)
    num_classes = 2
    input_size = 768 #+ 81

    best_model = load_best_model(model_path, num_classes, input_size, device)

    accuracy, precision, recall, f1 = evaluate(best_model, test_loader, device)
    print(f"Test Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")


test_file_path = "supremacist_test.csv"
test_feature_path = "supremacist_cls_test.npy"
batch_size = 16
model_path = "best_model(lstm).pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_model(test_file_path, test_feature_path, batch_size, model_path, device)